In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

import sys
from pathlib import Path

sys.path.append(str(Path("../src").resolve()))   # folder that CONTAINS the .py file

import utils
from utils import clean, get_data, get_results       # for src/my_utils.py


In [2]:
df_2024 = pd.read_csv("../data/df_2024_final.csv").assign(season=2024)
df_2025 = pd.read_csv("../data/df_2025_final.csv").assign(season=2025)
df = pd.concat([df_2024, df_2025], ignore_index=True)

In [3]:
df.head()
df= df.drop(columns="Unnamed: 0")

In [4]:

## necessary columns for each position to define separate data frames
id_cols = ['player_id', 'player', 'position', 'team', 'season', 'week',
           'date', 'opponent', 'atVs']

context_cols = ['opp_def_rank', 'team_pass_rank', 'team_rush_rank',
                'stadium', 'roof', 'surface', 'temp', 'wind', 'location',
                'gametime']

target_cols = ['fantasy_points', 'projected_points']



rushing = ['rushingAttempts', 'rushingYards', 'yardsPerRushAttempt',
           'rushingTouchdowns', 'longRushing']
receiving = ['receptions', 'receivingTargets', 'receivingYards',
             'yardsPerReception', 'receivingTouchdowns', 'longReception']
ball_security = ['fumbles', 'fumblesLost']
passing = ['completions', 'passingAttempts', 'passingYards', 'completionPct',
           'yardsPerPassAttempt', 'passingTouchdowns', 'interceptions',
           'longPassing', 'sacks', 'QBRating', 'adjQBR']
kicking = ['fieldGoalsMade', 'fieldGoalAttempts', 'fieldGoalPct',
           'longFieldGoalMade', 'fieldGoalsMadeYardsAverage',
           'fieldGoalsMade1_19', 'fieldGoalAttempts1_19',
           'fieldGoalsMade20_29', 'fieldGoalAttempts20_29',
           'fieldGoalsMade30_39', 'fieldGoalAttempts30_39',
           'fieldGoalsMade40_49', 'fieldGoalAttempts40_49',
           'fieldGoalsMade50', 'fieldGoalAttempts50',
           'extraPointsMade', 'extraPointAttempts', 'totalKickingPoints']

base = id_cols + context_cols + target_cols

position_cols = {
    'QB': base + passing + rushing + ball_security,
    'WR': base + receiving + rushing + ball_security,
    'TE': base + receiving + rushing + ball_security,
    'K':  base + kicking,
    "RB": base + rushing + receiving + ball_security
}
## generate each postional dataframe based on what scoring metrics contribute to their respective fantasy score
pos_dfs = {pos: df.loc[df['position'] == pos, cols].copy()
           for pos, cols in position_cols.items()}

qb_df, wr_df, te_df, k_df, rb_df = (pos_dfs[p] for p in ['QB', 'WR', 'TE', 'K', 'RB'])

In [5]:
## combined data frames based on cluster
qb_te = pd.concat([qb_df, te_df])
#qb_te.head()


In [6]:
wr_k = pd.concat([wr_df, k_df])

In [7]:
# now we need to sort each data frame in order
rb_df = rb_df.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)
qb_te = qb_te.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)
wr_k = wr_k.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)


In [8]:
## need to shift by -1 to create target of next weeks fantasy score
g_1 = rb_df.groupby(['player_id', "season"])
rb_df["target_fantasy_points"] = g_1["fantasy_points"].shift(-1)
rb_df['next_week_opponent'] = g_1['opponent'].shift(-1)
rb_df['next_week_atVs'] = g_1['atVs'].shift(-1)
rb_df['next_week_projected'] = g_1['projected_points'].shift(-1) 

g_2 = qb_te.groupby(['player_id', "season"])
qb_te["target_fantasy_points"] = g_2["fantasy_points"].shift(-1)
qb_te['next_week_opponent'] = g_2['opponent'].shift(-1)
qb_te['next_week_atVs'] = g_2['atVs'].shift(-1)
qb_te['next_week_projected'] = g_2['projected_points'].shift(-1) 

g_3 = wr_k.groupby(['player_id', "season"])
wr_k["target_fantasy_points"] = g_3["fantasy_points"].shift(-1)
wr_k["next_week_opponent"] = g_3["opponent"].shift(-1)
wr_k["next_week_atVs"] = g_3["atVs"].shift(-1)
wr_k["next_week_projected"] = g_3["projected_points"].shift(-1)
for col in context_cols:
    rb_df[f"next_week_{col}"] = g_1[col].shift(-1)
    qb_te[f"next_week_{col}"] = g_2[col].shift(-1)
    wr_k[f"next_week_{col}"] = g_3[col].shift(-1)


In [9]:
## encode categorical columns


shared_cat = ['position', 'next_week_opponent', 'next_week_atVs', 'next_week_surface',
                    'next_week_roof', 'next_week_location', 'next_week_stadium', 'next_week_gametime', "team"]
target = "target_fantasy_points"

rb_df[shared_cat] = rb_df[shared_cat].astype("category")
qb_te[shared_cat] = qb_te[shared_cat].astype("category")
wr_k[shared_cat] = wr_k[shared_cat].astype("category")

In [10]:
# generate training splits for each group
train_data_1 = rb_df[(rb_df["season"] == 2024) | ((rb_df["season"] == 2025) & (rb_df["week"] < 7))]
train_data_2 = qb_te[(qb_te["season"] == 2024) | ((qb_te["season"] == 2025) & (qb_te["week"] < 7))]
train_data_3 = wr_k[(wr_k["season"] == 2024) | ((wr_k["season"] == 2025) & (wr_k["week"] < 7))]

val_1 = rb_df[(rb_df["season"] == 2025) & (rb_df["week"].between(7,9))]
val_2 = qb_te[(qb_te["season"] == 2025) & (qb_te["week"].between(7,9))]
val_3 = wr_k[(wr_k["season"] == 2025) & (wr_k["week"].between(7,9))]

test_1 =  rb_df[(rb_df["season"] == 2025) & (rb_df["week"] >= 10)]
test_2 =  qb_te[(qb_te["season"] == 2025) & (qb_te["week"] >= 10)]
test_3 = wr_k[(wr_k["season"] ==2025) & (wr_k["week"] >= 10)]


In [11]:
## common columns across all the data frames
common = list(set(rb_df.columns) & set(qb_te.columns) & set(wr_k.columns))
irrelevant_columns = [
    'player_id', 'player', 'opponent', "date",
    'eventId', 'game_id', 'gameday', 'gametime', 'stadium', 'result',
    'score', 'fantasy_points', 'target_next_week_fantasy', 'espn',
    '_set', 'season',
    'med_proj_pts',
    'projected_points',
    'atVs', 'roof', 'surface', 'temp', 'wind', 'location',
    'opp_def_rank', 'team_pass_rank', 'team_rush_rank', 'next_week_projected' 
]

# want to get all irrelevant columns that dont contribute to score
irrelevant = list(set(common) & set(irrelevant_columns))

In [12]:


## Running back only
X1_train, y1_train = get_data(train_data_1, irrelevant, target)
X2_train, y2_train = get_data(train_data_2, irrelevant, target)
X3_train, y3_train = get_data(train_data_3, irrelevant, target)

##quarter back/ tight end
X1_val, y1_val = get_data(val_1, irrelevant, target)
X2_val, y2_val = get_data(val_2, irrelevant, target)
X3_val, y3_val = get_data(val_3, irrelevant, target)
## Wide reciever/kicker
X1_test, y1_test = get_data(test_1, irrelevant, target)
X2_test, y2_test = get_data(test_2, irrelevant, target)
X3_test, y3_test = get_data(test_3, irrelevant, target)

In [13]:
model1 = xgb.XGBRegressor(
    tree_method = "hist",
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
)



In [14]:
model1.fit(
    X1_train, y1_train,
    eval_set=[(X1_val, y1_val)],
    verbose=50 #how often to print validation score 
)

[0]	validation_0-rmse:8.21894
[50]	validation_0-rmse:6.80494
[100]	validation_0-rmse:5.68044
[150]	validation_0-rmse:4.84876
[200]	validation_0-rmse:4.13593
[250]	validation_0-rmse:3.58618
[300]	validation_0-rmse:3.12926
[350]	validation_0-rmse:2.73693
[400]	validation_0-rmse:2.43166
[450]	validation_0-rmse:2.20270
[500]	validation_0-rmse:2.03482
[550]	validation_0-rmse:1.86470
[600]	validation_0-rmse:1.71525
[650]	validation_0-rmse:1.61121
[700]	validation_0-rmse:1.53623
[750]	validation_0-rmse:1.45963
[800]	validation_0-rmse:1.40429
[850]	validation_0-rmse:1.36136
[900]	validation_0-rmse:1.32046
[950]	validation_0-rmse:1.28767
[1000]	validation_0-rmse:1.25557
[1050]	validation_0-rmse:1.22703
[1100]	validation_0-rmse:1.21042
[1150]	validation_0-rmse:1.19279
[1200]	validation_0-rmse:1.17872
[1250]	validation_0-rmse:1.16777
[1300]	validation_0-rmse:1.15888
[1350]	validation_0-rmse:1.14947
[1400]	validation_0-rmse:1.14076
[1450]	validation_0-rmse:1.13222
[1500]	validation_0-rmse:1.12582


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.7
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [15]:
model2 = xgb.XGBRegressor(
    tree_method = "hist",
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
)
model2.fit(X2_train, y2_train,
        eval_set=[(X2_val, y2_val)],
        verbose=50)

[0]	validation_0-rmse:8.31979
[50]	validation_0-rmse:6.74537
[100]	validation_0-rmse:5.53798
[150]	validation_0-rmse:4.58918
[200]	validation_0-rmse:3.80799
[250]	validation_0-rmse:3.17594
[300]	validation_0-rmse:2.71307
[350]	validation_0-rmse:2.34636
[400]	validation_0-rmse:2.02606
[450]	validation_0-rmse:1.82961
[500]	validation_0-rmse:1.64833
[550]	validation_0-rmse:1.47745
[600]	validation_0-rmse:1.33268
[650]	validation_0-rmse:1.20589
[700]	validation_0-rmse:1.11749
[750]	validation_0-rmse:1.04712
[800]	validation_0-rmse:0.98309
[850]	validation_0-rmse:0.94185
[900]	validation_0-rmse:0.90457
[950]	validation_0-rmse:0.87252
[1000]	validation_0-rmse:0.84745
[1050]	validation_0-rmse:0.82258
[1100]	validation_0-rmse:0.79782
[1150]	validation_0-rmse:0.77942
[1200]	validation_0-rmse:0.76444
[1250]	validation_0-rmse:0.75041
[1300]	validation_0-rmse:0.73647
[1350]	validation_0-rmse:0.72535
[1400]	validation_0-rmse:0.71593
[1450]	validation_0-rmse:0.70711
[1500]	validation_0-rmse:0.70062


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.7
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [16]:
model3 = xgb.XGBRegressor(
    tree_method = "hist",
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
)
model3.fit(X3_train, y3_train,
        eval_set=[(X3_val, y3_val)],
        verbose=50)

[0]	validation_0-rmse:6.82700
[50]	validation_0-rmse:5.60711
[100]	validation_0-rmse:4.66537
[150]	validation_0-rmse:3.81240
[200]	validation_0-rmse:3.15281
[250]	validation_0-rmse:2.63433
[300]	validation_0-rmse:2.26620
[350]	validation_0-rmse:1.88186
[400]	validation_0-rmse:1.63217
[450]	validation_0-rmse:1.46462
[500]	validation_0-rmse:1.28950
[550]	validation_0-rmse:1.14077
[600]	validation_0-rmse:1.01725
[650]	validation_0-rmse:0.91997
[700]	validation_0-rmse:0.82991
[750]	validation_0-rmse:0.75001
[800]	validation_0-rmse:0.68147
[850]	validation_0-rmse:0.63502
[900]	validation_0-rmse:0.60030
[950]	validation_0-rmse:0.56656
[1000]	validation_0-rmse:0.53379
[1050]	validation_0-rmse:0.50400
[1100]	validation_0-rmse:0.48009
[1150]	validation_0-rmse:0.46011
[1200]	validation_0-rmse:0.44030
[1250]	validation_0-rmse:0.42380
[1300]	validation_0-rmse:0.41135
[1350]	validation_0-rmse:0.39755
[1400]	validation_0-rmse:0.38731
[1450]	validation_0-rmse:0.37905
[1500]	validation_0-rmse:0.37096


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.7
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [17]:
## get our results for model 1
rb_results = get_results(rb_df, X1_test, y1_test, model1)

Model: MAE=0.424  RMSE=1.007  R2=0.985
ESPN: MAE=3.905  RMSE=5.852  R2=0.478


In [18]:
# model 2 results
qb_te_results=get_results(qb_te, X2_test, y2_test, model2)

Model: MAE=0.361  RMSE=0.722  R2=0.992
ESPN: MAE=3.716  RMSE=5.589  R2=0.513


In [19]:
wr_k_results = get_results(wr_k, X3_test, y3_test, model3)

Model: MAE=0.217  RMSE=0.454  R2=0.996
ESPN: MAE=3.968  RMSE=5.492  R2=0.378


In [20]:
all_pos = pd.concat([rb_results, qb_te_results, wr_k_results], ignore_index=True)
position_table = pd.crosstab(all_pos['position'], all_pos['winner'], margins=True)
position_table

winner,Model,espn,All
position,,,
K,232,4,236
QB,242,29,271
RB,644,131,775
TE,618,121,739
WR,1052,104,1156
All,2788,389,3177


## Modified Groups

Based on the findings in the ```Modeling_data_prep.ipynb``` we found the clusters with the most similar coefficient vectors. However, the cluster of Wide Reciever and Kicker makes no sense in terms of the game of football and based on our Spearman coefficients, Wide reciever was also close to QB. I would like to explore the potential cluster of WR, TE, WR and leave RB and K in their own seperate model.

In [13]:
#sort values to apply shifts
df1 = df.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)
# shift data
g = df1.groupby(['player_id', "season"])
df1["target_fantasy_points"] = g["fantasy_points"].shift(-1)
df1['next_week_opponent'] = g['opponent'].shift(-1)
df1['next_week_atVs'] = g['atVs'].shift(-1)
df1['next_week_projected'] = g['projected_points'].shift(-1)

In [18]:


#already have model for running back so we are going to leace that alone
qb, wr, te, k = df1[df1["position"] == "QB"], df1[df1["position"] == "WR"], df1[df1["position"] == "TE"], df1[df1["position"] == "K"]

# df2 = pd.concat([qb, wr, te])

# df2[shared_cat] = df2[shared_cat].astype("category")
# k[shared_cat] = k[shared_cat].astype("cateogory")

# training1 = df2[(df2["season"] == 2024) | ((df2["season"] == 2025) & (df2["week"] < 7))]
# val1 = df2[(df2["season"] == 2025) & (df2["week"].between(7,9))]
# test1 = df2[(df2["season"] == 2025) & (df2["week"] >= 10)]

# k_train = k[(k["season"] == 2024)|((k["season"]==2025) & (k["week"] < 7))]
# k_val = k[(k["season"] == 2025) & (k["week"].between(7, 9))]
# k_test = k[(k["season"] == 2025)& (k["week"] >= 10)]

# train_x, train_y = get_data(training1, irrelevant, target)
# val_x, val_y = get_data(val1, irrelevant, target)
# test_x, test_y = get_data(test1, irrelevant, target)

# train_k_x, train_k_y = get_data(k_train, irrelevant, target)
# val_k_x, val_k_y = get_data(k_val, irrelevant, target)
# test_k_x, test_k_y = get_data(k_test, irrelevant, target)

# # df2.head()

te.head()

,player_id,player,position,team,season,week,eventId,date,opponent,atVs,...,surface,temp,wind,location,team_pass_rank,team_rush_rank,target_fantasy_points,next_week_opponent,next_week_atVs,next_week_projected
36,9614,Marcedes Lewis,TE,CHI,2024,1,401671719,2024-09-08 17:00:00+00:00,TEN,home,...,grass,67.0,8.0,"Chicago,IL",NaN,NaN,0.0,HOU,away,0.0
37,9614,Marcedes Lewis,TE,CHI,2024,2,401671786,2024-09-16 00:20:00+00:00,HOU,away,...,astroturf,70.0,0.0,"Houston,TX",32.0,24.0,0.0,IND,away,0.0
38,9614,Marcedes Lewis,TE,CHI,2024,3,401671865,2024-09-22 17:00:00+00:00,IND,away,...,fieldturf,70.0,0.0,"Indianapolis,IN",31.0,28.0,0.0,LAR,home,0.0
39,9614,Marcedes Lewis,TE,CHI,2024,4,401671871,2024-09-29 17:00:00+00:00,LAR,home,...,grass,70.0,16.0,"Chicago,IL",19.0,31.0,0.0,CAR,home,0.0
40,9614,Marcedes Lewis,TE,CHI,2024,5,401671872,2024-10-06 17:00:00+00:00,CAR,home,...,grass,74.0,7.0,"Chicago,IL",24.0,29.0,1.2,JAX,home,0.0
